# Experiment: Building a Simple LLM Agent

## Aim

To build a simple **LLM Agent** that can use a registered calculator tool when required, execute the tool, and use the result to generate the final response.

## Objective

- Create a simple calculator function.
- Register the function as a tool.
- Provide the tool to an LLM agent.
- Craft queries that require tool execution.
- Understand how an agent decides when a tool is required.
- Execute the tool and use its result in the final response.
- Test the agent with different types of queries.


## 1. Concept

An **LLM Agent** combines:

```text
LLM + Tools + Agent Decision
```

For this experiment, the external tool is a **calculator function**.

### Example

User asks:

> What is 125 multiplied by 48?

The agent determines that a calculator tool is useful, invokes it, receives the result, and generates the final response.

```text
User Query
    ↓
LLM Agent
    ↓
Does the query require a tool?
    ↓
   YES
    ↓
Calculator Tool
    ↓
125 × 48
    ↓
6000
    ↓
LLM
    ↓
Final Answer
```

**Important:** The LLM decides when the tool is needed, while the Python calculator function performs the actual calculation.

## 2. Real-Time Example

The calculator tool will multiply two numbers.

Example query:

```text
What is 125 multiplied by 48?
```

The expected flow is:

```text
Operation → Multiplication
Tool      → calculator
Input 1   → 125
Input 2   → 48
Result    → 6000
```

## 3. System Architecture

```text
                    ┌─────────────────────┐
                    │        USER         │
                    │                     │
                    │ "What is 125 × 48?" │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    LANGCHAIN        │
                    │      AGENT          │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │        LLM          │
                    │                     │
                    │ Decide whether a    │
                    │ tool is required    │
                    └──────────┬──────────┘
                               │
                         Tool required
                               │
                               ▼
                    ┌─────────────────────┐
                    │   CALCULATOR TOOL   │
                    │                     │
                    │ calculator(125,48)  │
                    └──────────┬──────────┘
                               │
                               ▼
                             6000
                               │
                               ▼
                    ┌─────────────────────┐
                    │        LLM          │
                    │                     │
                    │ Generate final      │
                    │ response            │
                    └──────────┬──────────┘
                               │
                               ▼
                    "125 × 48 is 6000"
```

## 4. Software Requirements

- Python 3
- Jupyter Notebook / JupyterLab or VS Code with Jupyter support
- OpenAI API access
- LangChain
- LangChain OpenAI integration
- `python-dotenv`

## 5. Install Required Packages

Run the following cell once in the notebook:

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 6. Configure the API Key

Create a `.env` file in the same project folder as this notebook:

```env
OPENAI_API_KEY=your_api_key_here
```

Do not share or commit your API key to a public repository.

In [ ]:
import os

from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found. Check your .env file.")

print("API key loaded successfully.")

## 7. Register the Calculator Tool

The `@tool` decorator makes the Python function available to the LangChain agent as a tool.

In [ ]:
@tool
def calculator(a: float, b: float) -> float:
    """Multiply two numbers. Use this tool when the user asks for multiplication."""
    print(f"\nCalculator tool executed: {a} × {b}")
    return a * b

print("Tool registered:", calculator.name)
print("Tool description:", calculator.description)

## 8. Create the LLM

The LLM is responsible for understanding the user's request and deciding whether the registered tool is relevant.

In [ ]:
model = ChatOpenAI(
    model="gpt-5.6-luna"
)

print("LLM initialized successfully.")

## 9. Create the Agent

The agent is created by combining the LLM with the registered calculator tool.

In [ ]:
agent = create_agent(
    model=model,
    tools=[calculator]
)

print("Agent created successfully.")

## 10. Run the Agent

Test the agent with a query that requires calculator execution.

In [ ]:
user_question = "What is 125 multiplied by 48?"

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_question
            }
        ]
    }
)

print("\nFinal Answer:")
print(result["messages"][-1].content)

## 11. Execution Flow

For the query:

```text
What is 125 multiplied by 48?
```

The execution takes place as follows:

1. **User asks a question.**
2. **Agent receives the question.**
3. **LLM decides that a tool is required.**
4. **Calculator tool is invoked.**
5. **Python performs the calculation.**
6. **Tool result is returned to the agent.**
7. **LLM generates the final response.**

```text
User
 ↓
Agent
 ↓
LLM decides → Tool required
 ↓
calculator(125, 48)
 ↓
6000
 ↓
LLM
 ↓
125 multiplied by 48 is 6000.
```

## 12. Test Cases

Run the following queries by changing the value of `user_question` in the execution cell.

| Test Case | Query | Calculator Required? | Expected Behavior |
|---|---|---:|---|
| 1 | What is 25 × 40? | Yes | Calculator invoked |
| 2 | Calculate 123 × 17 | Yes | Calculator invoked |
| 3 | What is 125 × 48? | Yes | Calculator invoked |
| 4 | What is a calculator? | No | Normal LLM response |
| 5 | Explain multiplication | No | Normal LLM response |

## 13. Test Case 1

Query:

```text
What is 25 multiplied by 40?
```

Expected calculation:

```text
25 × 40 = 1000
```

In [ ]:
user_question = "What is 25 multiplied by 40?"

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": user_question}]
    }
)

print(result["messages"][-1].content)

## 14. Test Case 2

Query:

```text
Calculate 123 multiplied by 17.
```

Expected calculation:

```text
123 × 17 = 2091
```

In [ ]:
user_question = "Calculate 123 multiplied by 17."

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": user_question}]
    }
)

print(result["messages"][-1].content)

## 15. Test Case 3 — Tool Not Required

Query:

```text
What is a calculator?
```

The calculator tool is not necessary. The LLM should be able to answer directly.

In [ ]:
user_question = "What is a calculator?"

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": user_question}]
    }
)

print(result["messages"][-1].content)

## 16. Observation

During the experiment, observe:

1. Whether the agent identifies when the calculator is required.
2. Whether the calculator tool is invoked for multiplication queries.
3. Whether the correct input values are passed to the tool.
4. Whether the Python function produces the correct result.
5. Whether the result is incorporated into the final response.
6. Whether the agent avoids unnecessary calculator execution for unrelated questions.

## 17. LangChain vs Direct Function Calling

### Without LangChain

The application code handles more of the tool-calling workflow:

```text
Define Python function
        ↓
Define tool schema
        ↓
Send tool + user query to LLM
        ↓
Check LLM response
        ↓
Identify tool call
        ↓
Extract arguments
        ↓
Execute Python function
        ↓
Send result back to LLM
        ↓
Generate final response
```

### With LangChain

LangChain provides abstractions for the agent and tool workflow:

```text
@tool
   ↓
Register calculator
   ↓
create_agent()
   ↓
agent.invoke()
   ↓
Agent decides
   ↓
Tool executes
   ↓
Result returned
   ↓
Final response
```

### Key Advantage

LangChain reduces the amount of **manual orchestration code** required to connect the LLM, tools, tool execution, and agent workflow.

## 18. Result

The **Simple LLM Agent** was successfully implemented using LangChain with a calculator tool. The agent was tested with multiple queries and demonstrated the ability to invoke the calculator when required and use the tool result to generate the final response.

## 19. Key Learning

```text
Python Function
      ↓
Registered as a Tool
      ↓
Provided to the Agent
      ↓
LLM understands the User Query
      ↓
Agent decides whether the Tool is required
      ↓
Tool executes
      ↓
Result returned to the Agent
      ↓
LLM generates Final Response
```

> **An LLM Agent combines an LLM with external tools and can determine when a tool should be invoked to complete a task.**